<a href="https://colab.research.google.com/github/subhangidutta23-ship-it/ElderGuard/blob/main/ElderGuard_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers pandas scikit-learn

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/ElderGuard"
DATA_PATH = PROJECT_PATH + "/data"
MODEL_PATH = PROJECT_PATH + "/model/elderguard_distilbert"

print("ElderGuard folder exists:", os.path.exists(PROJECT_PATH))
print("Data folder exists:", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    print("\nFiles in data folder:")
    for file in os.listdir(DATA_PATH):
        print(" -", file)

ElderGuard folder exists: True
Data folder exists: True

Files in data folder:
 - fraud_dataset.csv
 - fraud_dataset.gsheet
 - spam.csv
 - English_NonScam.txt
 - English_Scam.txt
 - elderguard_combined.csv


In [ ]:
import os

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

print("Project folders are ready.")

Project folders are ready.


In [ ]:
import pandas as pd

# Original ElderGuard dataset
fraud = pd.read_csv(DATA_PATH + "/fraud_dataset.csv")

print("Original ElderGuard dataset")
print("---------------------------")
print("Rows:", len(fraud))
print("Columns:", fraud.columns.tolist())
print("\nLabels:")
print(fraud["label"].value_counts())

# SMS dataset
spam = pd.read_csv(
    DATA_PATH + "/spam.csv",
    encoding="latin-1"
)

print("\nSMS dataset")
print("-----------")
print("Rows:", len(spam))
print("Columns:", spam.columns.tolist())

# Scam file
with open(DATA_PATH + "/English_Scam.txt", "r", encoding="utf-8") as f:
    scam_text = f.read()

print("\nScam file loaded:", len(scam_text), "characters")

# Non-scam file
with open(DATA_PATH + "/English_NonScam.txt", "r", encoding="utf-8") as f:
    nonscam_text = f.read()

print("Non-scam file loaded:", len(nonscam_text), "characters")

Original ElderGuard dataset
---------------------------
Rows: 25
Columns: ['text', 'label']

Labels:
label
1    15
0    10
Name: count, dtype: int64

SMS dataset
-----------
Rows: 5572
Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

Scam file loaded: 160602 characters
Non-scam file loaded: 78839 characters


In [ ]:
import pandas as pd

# Original ElderGuard dataset
fraud = pd.read_csv(DATA_PATH + "/fraud_dataset.csv")

print("Original ElderGuard dataset")
print("---------------------------")
print("Rows:", len(fraud))
print("Columns:", fraud.columns.tolist())
print("\nLabels:")
print(fraud["label"].value_counts())

# SMS dataset
spam = pd.read_csv(
    DATA_PATH + "/spam.csv",
    encoding="latin-1"
)

print("\nSMS dataset")
print("-----------")
print("Rows:", len(spam))
print("Columns:", spam.columns.tolist())

# Scam file
with open(DATA_PATH + "/English_Scam.txt", "r", encoding="utf-8") as f:
    scam_text = f.read()

print("\nScam file loaded:", len(scam_text), "characters")

# Non-scam file
with open(DATA_PATH + "/English_NonScam.txt", "r", encoding="utf-8") as f:
    nonscam_text = f.read()

print("Non-scam file loaded:", len(nonscam_text), "characters")

Original ElderGuard dataset
---------------------------
Rows: 25
Columns: ['text', 'label']

Labels:
label
1    15
0    10
Name: count, dtype: int64

SMS dataset
-----------
Rows: 5572
Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

Scam file loaded: 160602 characters
Non-scam file loaded: 78839 characters


In [ ]:
# STEP 8 — Load combined dataset and split into train/test

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Define the project paths again
PROJECT_PATH = "/content/drive/MyDrive/ElderGuard"
DATA_PATH = os.path.join(PROJECT_PATH, "data")
COMBINED_PATH = os.path.join(DATA_PATH, "elderguard_combined.csv")

# Check that the file exists
print("Looking for dataset at:")
print(COMBINED_PATH)

if not os.path.exists(COMBINED_PATH):
    raise FileNotFoundError(
        "elderguard_combined.csv was not found. "
        "Please run STEP 7 first."
    )

# Load the combined dataset
df = pd.read_csv(COMBINED_PATH)

# Remove missing values
df = df.dropna(subset=["text", "label"])

# Make sure labels are integers
df["label"] = df["label"].astype(int)

print("\nTotal examples:", len(df))
print("\nLabel distribution:")
print(df["label"].value_counts())

# Split into training and testing data
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTraining examples:", len(train_df))
print("Testing examples:", len(test_df))

print("\nSTEP 8 completed successfully!")

Looking for dataset at:
/content/drive/MyDrive/ElderGuard/data/elderguard_combined.csv

Total examples: 5980

Label distribution:
label
0    4926
1    1054
Name: count, dtype: int64

Training examples: 4784
Testing examples: 1196

STEP 8 completed successfully!


In [ ]:
# STEP 9 — Load DistilBERT tokenizer and model

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading DistilBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("\nDistilBERT loaded successfully!")
print("Number of labels:", model.config.num_labels)

Loading tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading DistilBERT model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



DistilBERT loaded successfully!
Number of labels: 2


In [ ]:
# STEP 10 — Create PyTorch Dataset

import torch
from torch.utils.data import Dataset

class ElderGuardDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


# Create training and testing datasets
train_dataset = ElderGuardDataset(
    train_df,
    tokenizer
)

test_dataset = ElderGuardDataset(
    test_df,
    tokenizer
)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))
print("STEP 10 completed successfully!")

Training dataset: 4784
Testing dataset: 1196
STEP 10 completed successfully!


In [ ]:
# STEP 11 — Create DataLoaders

from torch.utils.data import DataLoader

# Batch size for T4 GPU
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))
print("STEP 11 completed successfully!")

Batch size: 32
Training batches: 150
Testing batches: 38
STEP 11 completed successfully!


In [ ]:
# STEP 12 — Move model to GPU

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("WARNING: GPU not detected. Using CPU.")

model = model.to(device)

print("Model is now using:", device)
print("STEP 12 completed successfully!")

GPU detected: Tesla T4
Model is now using: cuda
STEP 12 completed successfully!


In [ ]:
# STEP 13 — Train ElderGuard DistilBERT model

from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

# Number of training rounds
EPOCHS = 3

print("Starting training...")
print("Epochs:", EPOCHS)
print("Training examples:", len(train_dataset))
print("Device:", device)
print()

for epoch in range(EPOCHS):

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()

    total_loss = 0

    for batch_number, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        total_loss += loss.item()

        # Show progress every 25 batches
        if (batch_number + 1) % 25 == 0:
            print(
                f"Epoch {epoch + 1}/{EPOCHS} "
                f"- Batch {batch_number + 1}/{len(train_loader)} "
                f"- Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(train_loader)

    # -------------------------
    # EVALUATION
    # -------------------------
    model.eval()

    predictions = []
    actual_labels = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predicted_labels = torch.argmax(
                outputs.logits,
                dim=1
            )

            predictions.extend(
                predicted_labels.cpu().numpy()
            )

            actual_labels.extend(
                labels.cpu().numpy()
            )

    # Calculate metrics
    accuracy = accuracy_score(
        actual_labels,
        predictions
    )

    precision = precision_score(
        actual_labels,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        actual_labels,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        actual_labels,
        predictions,
        zero_division=0
    )

    print()
    print("=" * 50)
    print(f"Epoch {epoch + 1} completed")
    print(f"Average training loss: {average_loss:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("=" * 50)
    print()

print("TRAINING COMPLETED SUCCESSFULLY!")

Starting training...
Epochs: 3
Training examples: 4784
Device: cuda

Epoch 1/3 - Batch 25/150 - Loss: 0.2285
Epoch 1/3 - Batch 50/150 - Loss: 0.0205
Epoch 1/3 - Batch 75/150 - Loss: 0.2708
Epoch 1/3 - Batch 100/150 - Loss: 0.1248
Epoch 1/3 - Batch 125/150 - Loss: 0.1422
Epoch 1/3 - Batch 150/150 - Loss: 0.0167

Epoch 1 completed
Average training loss: 0.1385
Accuracy:  0.9841
Precision: 0.9706
Recall:    0.9384
F1 Score:  0.9542

Epoch 2/3 - Batch 25/150 - Loss: 0.0173
Epoch 2/3 - Batch 50/150 - Loss: 0.0046
Epoch 2/3 - Batch 75/150 - Loss: 0.0071
Epoch 2/3 - Batch 100/150 - Loss: 0.0169
Epoch 2/3 - Batch 125/150 - Loss: 0.0059
Epoch 2/3 - Batch 150/150 - Loss: 0.0601

Epoch 2 completed
Average training loss: 0.0330
Accuracy:  0.9891
Precision: 0.9806
Recall:    0.9573
F1 Score:  0.9688

Epoch 3/3 - Batch 25/150 - Loss: 0.0026
Epoch 3/3 - Batch 50/150 - Loss: 0.0042
Epoch 3/3 - Batch 75/150 - Loss: 0.0019
Epoch 3/3 - Batch 100/150 - Loss: 0.0044
Epoch 3/3 - Batch 125/150 - Loss: 0.0043

In [ ]:
# STEP 14 — Save trained ElderGuard model

import os

PROJECT_PATH = "/content/drive/MyDrive/ElderGuard"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "model",
    "elderguard_distilbert"
)

os.makedirs(MODEL_PATH, exist_ok=True)

# Save trained model
model.save_pretrained(MODEL_PATH)

# Save tokenizer
tokenizer.save_pretrained(MODEL_PATH)

print("Model saved to:")
print(MODEL_PATH)

print("\nFiles inside model folder:")

for filename in sorted(os.listdir(MODEL_PATH)):
    file_path = os.path.join(MODEL_PATH, filename)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"{filename} | {size_mb:.2f} MB")

print("\nSTEP 14 completed successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:
/content/drive/MyDrive/ElderGuard/model/elderguard_distilbert

Files inside model folder:
checkpoint-10 | 0.00 MB
checkpoint-15 | 0.00 MB
checkpoint-20 | 0.00 MB
checkpoint-25 | 0.00 MB
checkpoint-5 | 0.00 MB
config.json | 0.00 MB
model.safetensors | 255.43 MB
tokenizer.json | 0.68 MB
tokenizer_config.json | 0.00 MB
training_args.bin | 0.00 MB

STEP 14 completed successfully!


In [ ]:
# STEP 15 — Verify saved model

import os

model_file = os.path.join(
    MODEL_PATH,
    "model.safetensors"
)

if os.path.exists(model_file):

    size_mb = os.path.getsize(model_file) / (1024 * 1024)

    print("SUCCESS!")
    print("model.safetensors exists.")
    print(f"Model size: {size_mb:.2f} MB")

else:

    raise FileNotFoundError(
        "model.safetensors was not found. "
        "The model was not saved correctly."
    )

print("\nSTEP 15 completed successfully!")

SUCCESS!
model.safetensors exists.
Model size: 255.43 MB

STEP 15 completed successfully!


In [ ]:
# STEP 16 — Reload model from Google Drive

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading saved tokenizer...")

saved_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

print("Loading saved model...")

saved_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

saved_model = saved_model.to(device)
saved_model.eval()

print("\nSaved model loaded successfully!")
print("Device:", device)
print("Number of labels:", saved_model.config.num_labels)

print("\nSTEP 16 completed successfully!")

Loading saved tokenizer...
Loading saved model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Saved model loaded successfully!
Device: cuda
Number of labels: 2

STEP 16 completed successfully!


In [ ]:
# STEP 17 — ElderGuard prediction function

import torch

def predict_message(message):

    # Convert text into tokens
    inputs = saved_tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Make prediction
    with torch.no_grad():

        outputs = saved_model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

        prediction = torch.argmax(
            probabilities,
            dim=1
        ).item()

        confidence = (
            probabilities[0, prediction].item() * 100
        )

    # Convert numerical label to text
    if prediction == 1:
        label = "FRAUD"
    else:
        label = "SAFE"

    print("Message:")
    print(message)

    print("\nPrediction:", label)
    print(f"Confidence: {confidence:.2f}%")

    return label, confidence

print("STEP 17 completed successfully!")

STEP 17 completed successfully!


In [ ]:
# STEP 18 — Test ElderGuard

print("TEST 1")
predict_message(
    "This is CBI. Your Aadhaar is involved in an illegal case. Transfer money immediately."
)

print("\n" + "=" * 60 + "\n")

print("TEST 2")
predict_message(
    "Your electricity bill is due tomorrow."
)

print("\n" + "=" * 60 + "\n")

print("TEST 3")
predict_message(
    "You are under digital arrest. Do not disconnect this call."
)

TEST 1
Message:
This is CBI. Your Aadhaar is involved in an illegal case. Transfer money immediately.

Prediction: FRAUD
Confidence: 99.09%


TEST 2
Message:
Your electricity bill is due tomorrow.

Prediction: SAFE
Confidence: 99.92%


TEST 3
Message:
You are under digital arrest. Do not disconnect this call.

Prediction: FRAUD
Confidence: 98.82%


('FRAUD', 98.81914258003235)

In [ ]:
# STEP 19 — Test ElderGuard on different scam scenarios

test_messages = {

    "Digital Arrest":
        "This is CBI. Your Aadhaar is connected to a money laundering case. "
        "You are under digital arrest. Do not disconnect the call.",

    "Investment Scam":
        "Our expert trading team can guarantee 100% returns. "
        "Transfer Rs 50000 today to activate your premium investment account.",

    "KYC Scam":
        "Your bank account will be frozen today unless you complete your KYC "
        "immediately. Install this remote access application and share your OTP.",

    "Safe Bank Message":
        "Your monthly bank statement is now available. "
        "Please log in through your usual banking application to view it.",

    "Safe Delivery Message":
        "Your package has been shipped and will arrive tomorrow.",

    "Safe Appointment":
        "This is a reminder that your hospital appointment is scheduled for tomorrow."
}

for category, message in test_messages.items():

    print("\n" + "=" * 70)
    print(category)
    print("=" * 70)

    predict_message(message)


Digital Arrest
Message:
This is CBI. Your Aadhaar is connected to a money laundering case. You are under digital arrest. Do not disconnect the call.

Prediction: FRAUD
Confidence: 98.96%

Investment Scam
Message:
Our expert trading team can guarantee 100% returns. Transfer Rs 50000 today to activate your premium investment account.

Prediction: FRAUD
Confidence: 99.81%

KYC Scam
Message:
Your bank account will be frozen today unless you complete your KYC immediately. Install this remote access application and share your OTP.

Prediction: FRAUD
Confidence: 99.62%

Safe Bank Message
Message:
Your monthly bank statement is now available. Please log in through your usual banking application to view it.

Prediction: FRAUD
Confidence: 98.36%

Safe Delivery Message
Message:
Your package has been shipped and will arrive tomorrow.

Prediction: SAFE
Confidence: 99.59%

Safe Appointment
Message:
This is a reminder that your hospital appointment is scheduled for tomorrow.

Prediction: SAFE
Confid

In [ ]:
# STEP 20 — ElderGuard Risk Scoring

SCAM_INDICATORS = {

    "digital_arrest": [
        "digital arrest",
        "under arrest",
        "do not disconnect",
        "do not tell anyone"
    ],

    "authority_impersonation": [
        "cbi",
        "police",
        "customs",
        "enforcement directorate",
        "ed officer",
        "government officer"
    ],

    "investment_scam": [
        "guaranteed return",
        "100% returns",
        "double your money",
        "guaranteed profit",
        "investment opportunity"
    ],

    "financial_pressure": [
        "transfer money",
        "send money",
        "pay immediately",
        "verification fee",
        "processing fee",
        "rbi account"
    ],

    "remote_access": [
        "anydesk",
        "teamviewer",
        "remote access",
        "screen sharing",
        "install this application"
    ],

    "kyc_pressure": [
        "kyc",
        "account will be frozen",
        "account will be blocked",
        "update immediately"
    ],

    "credential_request": [
        "otp",
        "pin",
        "password",
        "bank details",
        "card details"
    ]
}


def calculate_risk(message):

    # Get model prediction
    inputs = saved_tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = saved_model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

    fraud_probability = probabilities[0][1].item()

    # Start risk score from model probability
    risk_score = fraud_probability * 60

    message_lower = message.lower()

    detected_indicators = []

    # Check scam indicators
    for category, keywords in SCAM_INDICATORS.items():

        for keyword in keywords:

            if keyword in message_lower:

                detected_indicators.append(
                    (category, keyword)
                )

                risk_score += 5

                break

    # Maximum score
    risk_score = min(risk_score, 100)

    # Risk level
    if risk_score >= 70:

        risk_level = "HIGH"

    elif risk_score >= 40:

        risk_level = "MEDIUM"

    else:

        risk_level = "LOW"

    return {
        "fraud_probability": fraud_probability * 100,
        "risk_score": risk_score,
        "risk_level": risk_level,
        "indicators": detected_indicators
    }


print("STEP 20 function created successfully!")

STEP 20 function created successfully!


In [ ]:
# STEP 20 TEST

message = (
    "This is CBI. You are under digital arrest. "
    "Do not disconnect. Transfer money immediately."
)

result = calculate_risk(message)

print("Message:")
print(message)

print("\nFraud probability:")
print(f"{result['fraud_probability']:.2f}%")

print("\nElderGuard risk score:")
print(f"{result['risk_score']:.2f}/100")

print("\nRisk level:")
print(result["risk_level"])

print("\nDetected indicators:")

for category, keyword in result["indicators"]:
    print(f"- {category}: {keyword}")

Message:
This is CBI. You are under digital arrest. Do not disconnect. Transfer money immediately.

Fraud probability:
99.37%

ElderGuard risk score:
74.62/100

Risk level:
HIGH

Detected indicators:
- digital_arrest: digital arrest
- authority_impersonation: cbi
- financial_pressure: transfer money


In [ ]:
# STEP 21 — ElderGuard Transaction Time-Lock

import time
from datetime import datetime, timedelta


COOLDOWN_MINUTES = 15


def transaction_check(amount, risk_score):

    print("\n" + "=" * 60)
    print("ELDERGUARD TRANSACTION SECURITY")
    print("=" * 60)

    print(f"Transaction amount: ₹{amount}")
    print(f"Risk score: {risk_score:.2f}/100")

    if risk_score >= 70:

        unlock_time = datetime.now() + timedelta(
            minutes=COOLDOWN_MINUTES
        )

        print("\n⚠ HIGH-RISK TRANSACTION DETECTED")
        print("Transaction has been temporarily locked.")
        print(f"Cool-off period: {COOLDOWN_MINUTES} minutes")
        print(
            "Transaction can be reviewed after:",
            unlock_time.strftime("%H:%M:%S")
        )

        return "LOCKED"

    elif risk_score >= 40:

        print("\n⚠ MEDIUM-RISK TRANSACTION")
        print("Additional confirmation required.")

        return "CONFIRMATION_REQUIRED"

    else:

        print("\n✓ LOW-RISK TRANSACTION")
        print("No ElderGuard time-lock triggered.")

        return "ALLOWED"

In [ ]:
# STEP 21 TEST

status = transaction_check(
    amount=50000,
    risk_score=92
)

print("\nTransaction status:", status)


ELDERGUARD TRANSACTION SECURITY
Transaction amount: ₹50000
Risk score: 92.00/100

⚠ HIGH-RISK TRANSACTION DETECTED
Transaction has been temporarily locked.
Cool-off period: 15 minutes
Transaction can be reviewed after: 13:52:47

Transaction status: LOCKED


In [ ]:
# STEP 22 — Guardian Alert Prototype


def guardian_alert(
    risk_score,
    risk_level,
    transaction_amount,
    indicators
):

    if risk_level == "HIGH":

        print("\n" + "=" * 60)
        print("🚨 ELDERGUARD GUARDIAN ALERT")
        print("=" * 60)

        print("A high-risk activity was detected.")

        print(f"\nRisk score: {risk_score:.2f}/100")
        print(f"Transaction amount: ₹{transaction_amount}")

        print("\nDetected indicators:")

        for category, keyword in indicators:
            print(f"- {category}: {keyword}")

        print("\nRecommended action:")
        print("Transaction temporarily locked.")
        print("Guardian confirmation required.")

        return True

    return False

In [ ]:
# STEP 22 TEST

message = (
    "This is CBI. You are under digital arrest. "
    "Do not disconnect. Transfer money immediately."
)

result = calculate_risk(message)

guardian_alert(
    risk_score=result["risk_score"],
    risk_level=result["risk_level"],
    transaction_amount=50000,
    indicators=result["indicators"]
)


🚨 ELDERGUARD GUARDIAN ALERT
A high-risk activity was detected.

Risk score: 74.62/100
Transaction amount: ₹50000

Detected indicators:
- digital_arrest: digital arrest
- authority_impersonation: cbi
- financial_pressure: transfer money

Recommended action:
Transaction temporarily locked.
Guardian confirmation required.


True

In [ ]:
!pip install -q librosa soundfile

In [ ]:
# STEP 23 — Voice analysis setup

import librosa
import numpy as np

print("Librosa version:", librosa.__version__)
print("Voice analysis libraries loaded successfully!")

Librosa version: 0.11.0
Voice analysis libraries loaded successfully!


In [ ]:
# STEP 23C — Extract voice features

def extract_voice_features(audio_path):

    audio, sample_rate = librosa.load(
        audio_path,
        sr=16000
    )

    # MFCC features
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=13
    )

    # Energy
    rms = librosa.feature.rms(
        y=audio
    )

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(
        audio
    )

    features = {

        "mean_mfcc": np.mean(mfcc),

        "mean_energy": np.mean(rms),

        "mean_zcr": np.mean(zcr)
    }

    return features

In [ ]:
# STEP 24 — Multimodal risk fusion

def combined_risk_score(
    text_fraud_probability,
    voice_stress_probability
):

    text_score = text_fraud_probability * 0.70

    voice_score = voice_stress_probability * 0.30

    final_score = (
        text_score +
        voice_score
    )

    return final_score

In [ ]:
# STEP 24 TEST

text_probability = 95
voice_probability = 80

final_score = combined_risk_score(
    text_probability,
    voice_probability
)

print("Text fraud probability:", text_probability)
print("Voice stress probability:", voice_probability)
print("Combined ElderGuard risk:", final_score)

Text fraud probability: 95
Voice stress probability: 80
Combined ElderGuard risk: 90.5


In [ ]:
# STEP 25 — Synthetic transaction scenarios

import random
import pandas as pd


def generate_transaction():

    amount = random.choice([
        5000,
        10000,
        25000,
        50000,
        100000
    ])

    transaction_type = random.choice([
        "UPI",
        "RTGS"
    ])

    risk_level = random.choice([
        "LOW",
        "MEDIUM",
        "HIGH"
    ])

    stress_level = random.choice([
        "LOW",
        "MEDIUM",
        "HIGH"
    ])

    return {
        "amount": amount,
        "transaction_type": transaction_type,
        "risk_level": risk_level,
        "stress_level": stress_level
    }


synthetic_data = []

for _ in range(1000):

    synthetic_data.append(
        generate_transaction()
    )


synthetic_df = pd.DataFrame(
    synthetic_data
)

print(synthetic_df.head())
print("\nTotal synthetic scenarios:", len(synthetic_df))

   amount transaction_type risk_level stress_level
0   10000             RTGS     MEDIUM         HIGH
1   25000              UPI     MEDIUM         HIGH
2   25000             RTGS     MEDIUM          LOW
3   10000             RTGS       HIGH          LOW
4   50000              UPI     MEDIUM          LOW

Total synthetic scenarios: 1000


In [ ]:
# STEP 26 — Complete ElderGuard Decision Engine

def elderguard_analyze(message, transaction_amount=0):

    print("\n" + "=" * 70)
    print("                 ELDERGUARD ANALYSIS")
    print("=" * 70)

    print("\nMessage:")
    print(message)

    # ---------------------------------
    # 1. Calculate fraud/risk score
    # ---------------------------------

    result = calculate_risk(message)

    fraud_probability = result["fraud_probability"]
    risk_score = result["risk_score"]
    risk_level = result["risk_level"]
    indicators = result["indicators"]

    # ---------------------------------
    # 2. Display AI result
    # ---------------------------------

    print("\nAI FRAUD PROBABILITY:")
    print(f"{fraud_probability:.2f}%")

    print("\nELDERGUARD RISK SCORE:")
    print(f"{risk_score:.2f}/100")

    print("\nRISK LEVEL:")
    print(risk_level)

    # ---------------------------------
    # 3. Display detected indicators
    # ---------------------------------

    print("\nDETECTED INDICATORS:")

    if indicators:

        for category, keyword in indicators:
            print(f"- {category}: {keyword}")

    else:

        print("- No specific scam indicators detected.")

    # ---------------------------------
    # 4. Transaction decision
    # ---------------------------------

    print("\nTRANSACTION DECISION:")

    if transaction_amount > 0:

        status = transaction_check(
            transaction_amount,
            risk_score
        )

    else:

        if risk_level == "HIGH":

            status = "HIGH_RISK_WARNING"

        elif risk_level == "MEDIUM":

            status = "CONFIRMATION_REQUIRED"

        else:

            status = "SAFE"

    # ---------------------------------
    # 5. Guardian alert
    # ---------------------------------

    if risk_level == "HIGH":

        guardian_alert(
            risk_score=risk_score,
            risk_level=risk_level,
            transaction_amount=transaction_amount,
            indicators=indicators
        )

    # ---------------------------------
    # 6. Final result
    # ---------------------------------

    print("\n" + "=" * 70)
    print("FINAL ELDERGUARD DECISION")
    print("=" * 70)

    print("Risk level:", risk_level)
    print("Transaction status:", status)

    return {
        "fraud_probability": fraud_probability,
        "risk_score": risk_score,
        "risk_level": risk_level,
        "transaction_status": status,
        "indicators": indicators
    }


print("STEP 26 completed successfully!")

STEP 26 completed successfully!


In [ ]:
# STEP 26 TEST — High-risk Digital Arrest scenario

message = (
    "This is CBI. Your Aadhaar is involved in an illegal case. "
    "You are under digital arrest. Do not disconnect the call. "
    "Transfer Rs 50000 immediately to the RBI verification account."
)

result = elderguard_analyze(
    message,
    transaction_amount=50000
)


                 ELDERGUARD ANALYSIS

Message:
This is CBI. Your Aadhaar is involved in an illegal case. You are under digital arrest. Do not disconnect the call. Transfer Rs 50000 immediately to the RBI verification account.

AI FRAUD PROBABILITY:
99.78%

ELDERGUARD RISK SCORE:
69.87/100

RISK LEVEL:
MEDIUM

DETECTED INDICATORS:
- digital_arrest: digital arrest
- authority_impersonation: cbi

TRANSACTION DECISION:

ELDERGUARD TRANSACTION SECURITY
Transaction amount: ₹50000
Risk score: 69.87/100

⚠ MEDIUM-RISK TRANSACTION
Additional confirmation required.

FINAL ELDERGUARD DECISION
Risk level: MEDIUM
Transaction status: CONFIRMATION_REQUIRED


In [ ]:
# STEP 26 TEST — Safe scenario

message = (
    "Your electricity bill is due tomorrow. "
    "Please pay it through your normal electricity provider."
)

result = elderguard_analyze(
    message,
    transaction_amount=1000
)


                 ELDERGUARD ANALYSIS

Message:
Your electricity bill is due tomorrow. Please pay it through your normal electricity provider.

AI FRAUD PROBABILITY:
29.06%

ELDERGUARD RISK SCORE:
17.44/100

RISK LEVEL:
LOW

DETECTED INDICATORS:
- No specific scam indicators detected.

TRANSACTION DECISION:

ELDERGUARD TRANSACTION SECURITY
Transaction amount: ₹1000
Risk score: 17.44/100

✓ LOW-RISK TRANSACTION
No ElderGuard time-lock triggered.

FINAL ELDERGUARD DECISION
Risk level: LOW
Transaction status: ALLOWED


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

saved_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = saved_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())


# Get the actual labels directly from test_df
all_labels = test_df["label"].tolist()


# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print("=" * 60)
print("ELDERGUARD MODEL EVALUATION")
print("=" * 60)

print(f"\nAccuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1 Score  : {f1 * 100:.2f}%")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["SAFE", "FRAUD"]
    )
)

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(all_labels, all_predictions)

print(cm)

ELDERGUARD MODEL EVALUATION

Accuracy  : 99.00%
Precision : 98.07%
Recall    : 96.21%
F1 Score  : 97.13%

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        SAFE       0.99      1.00      0.99       985
       FRAUD       0.98      0.96      0.97       211

    accuracy                           0.99      1196
   macro avg       0.99      0.98      0.98      1196
weighted avg       0.99      0.99      0.99      1196


CONFUSION MATRIX
[[981   4]
 [  8 203]]


In [ ]:
# Create a copy of the test data
error_analysis = test_df.copy()

# Add model predictions
error_analysis["prediction"] = all_predictions

# Convert numbers to readable labels
error_analysis["actual_label"] = error_analysis["label"].map({
    0: "SAFE",
    1: "FRAUD"
})

error_analysis["predicted_label"] = error_analysis["prediction"].map({
    0: "SAFE",
    1: "FRAUD"
})

# Keep only incorrectly classified messages
errors = error_analysis[
    error_analysis["label"] != error_analysis["prediction"]
]

print("=" * 70)
print("ELDERGUARD ERROR ANALYSIS")
print("=" * 70)

print("\nTotal incorrect predictions:", len(errors))

for index, row in errors.iterrows():

    print("\n" + "-" * 70)
    print("MESSAGE:")
    print(row["text"])

    print("\nACTUAL:", row["actual_label"])
    print("PREDICTED:", row["predicted_label"])

ELDERGUARD ERROR ANALYSIS

Total incorrect predictions: 12

----------------------------------------------------------------------
MESSAGE:
They have a thread on the wishlist section of the forums where ppl post nitro requests. Start from the last page and collect from the bottom up.

ACTUAL: SAFE
PREDICTED: FRAUD

----------------------------------------------------------------------
MESSAGE:
[Greetings], this is [Name]. I saw a notice regarding delays in the latest stimulus payments. To avoid confusion or misinformation, it is best to visit your bank's nearest branch to confirm your disbursement status. Bring your account details and any official communication received.

ACTUAL: SAFE
PREDICTED: FRAUD

----------------------------------------------------------------------
MESSAGE:
Hello. We need some posh birds and chaps to user trial prods for champneys. Can i put you down? I need your address and dob asap. Ta r

ACTUAL: FRAUD
PREDICTED: SAFE

----------------------------------------

In [ ]:
elderguard_test_messages = {

    "Digital Arrest": [
        "This is CBI calling. Your Aadhaar has been linked to a money laundering case. You are under digital arrest. Do not disconnect the call.",
        "Your identity has been used in an illegal transaction. Stay on video call and transfer money to the RBI verification account.",
        "I am calling from the police department. Your SIM is involved in a criminal case. Do not tell anyone about this call."
    ],

    "Investment Scam": [
        "Our expert trading group guarantees 100% returns. Transfer ₹50000 today to activate your premium investment account.",
        "You can double your money in just seven days. Pay the processing fee now to receive your investment profit.",
        "This is a guaranteed investment opportunity. Send ₹25000 immediately and our team will manage your account."
    ],

    "KYC Scam": [
        "Your bank account will be frozen today unless you complete your KYC immediately. Share your OTP to verify your account.",
        "Your KYC has expired. Send your Aadhaar number and bank details immediately or your account will be blocked.",
        "Complete your KYC verification now by clicking this link or your pension account will be suspended."
    ],

    "Remote Access Scam": [
        "Install AnyDesk so our bank officer can access your screen and verify your account.",
        "Download this remote access application immediately so we can fix your banking problem.",
        "Please install TeamViewer and share your screen so our support team can verify the transaction."
    ],

    "OTP Credential Scam": [
        "Your bank account has a suspicious transaction. Tell me the OTP you received so I can cancel it.",
        "Please provide your ATM PIN and OTP to complete the verification process.",
        "Send your card number, PIN and password immediately to prevent your account from being blocked."
    ],

    "Payment Pressure": [
        "Transfer the money immediately to the verification account or legal action will be taken against you.",
        "Pay the processing fee now or your bank account will be permanently frozen.",
        "Send ₹10000 immediately to release your blocked account."
    ],

    "Safe Banking": [
        "Your monthly bank statement is available. Please check it through your usual banking application.",
        "Your salary has been credited to your bank account. You can check the transaction in your banking app.",
        "Your bank branch will be closed on Sunday. Normal banking services will resume on Monday."
    ],

    "Safe Bills": [
        "Your electricity bill is due tomorrow. Please pay it through your normal electricity provider.",
        "Your internet bill is due this week. You can make the payment using your usual provider.",
        "Your water bill has been generated. Please check the amount through the official municipal website."
    ],

    "Safe Delivery": [
        "Your package has been shipped and will arrive tomorrow.",
        "Your delivery is scheduled for Friday between 10 AM and 2 PM.",
        "Your parcel has reached the local delivery centre."
    ],

    "Safe Personal": [
        "Your hospital appointment is scheduled for tomorrow at 10 AM.",
        "We are meeting for lunch tomorrow at the usual restaurant.",
        "Please call me when you reach home safely."
    ]
}

print("ElderGuard test categories:", len(elderguard_test_messages))

total_messages = sum(
    len(messages)
    for messages in elderguard_test_messages.values()
)

print("Total test messages:", total_messages)

ElderGuard test categories: 10
Total test messages: 30


In [ ]:
print("=" * 80)
print("ELDERGUARD-SPECIFIC MODEL TEST")
print("=" * 80)

elderguard_results = []

for category, messages in elderguard_test_messages.items():

    print("\n" + "=" * 80)
    print(category)
    print("=" * 80)

    for message in messages:

        # Tokenize message
        inputs = saved_tokenizer(
            message,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        # Move inputs to GPU/CPU
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # Model prediction
        with torch.no_grad():
            outputs = saved_model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=1)

        safe_probability = probabilities[0][0].item() * 100
        fraud_probability = probabilities[0][1].item() * 100

        prediction = "FRAUD" if fraud_probability >= safe_probability else "SAFE"

        print("\nMessage:")
        print(message)

        print(f"\nSAFE probability : {safe_probability:.2f}%")
        print(f"FRAUD probability: {fraud_probability:.2f}%")
        print(f"PREDICTION       : {prediction}")

        elderguard_results.append({
            "category": category,
            "message": message,
            "safe_probability": safe_probability,
            "fraud_probability": fraud_probability,
            "prediction": prediction
        })

print("\n" + "=" * 80)
print("TESTING COMPLETED")
print("=" * 80)

print("Total messages tested:", len(elderguard_results))

ELDERGUARD-SPECIFIC MODEL TEST

Digital Arrest

Message:
This is CBI calling. Your Aadhaar has been linked to a money laundering case. You are under digital arrest. Do not disconnect the call.

SAFE probability : 1.04%
FRAUD probability: 98.96%
PREDICTION       : FRAUD

Message:
Your identity has been used in an illegal transaction. Stay on video call and transfer money to the RBI verification account.

SAFE probability : 0.26%
FRAUD probability: 99.74%
PREDICTION       : FRAUD

Message:
I am calling from the police department. Your SIM is involved in a criminal case. Do not tell anyone about this call.

SAFE probability : 8.54%
FRAUD probability: 91.46%
PREDICTION       : FRAUD

Investment Scam

Message:
Our expert trading group guarantees 100% returns. Transfer ₹50000 today to activate your premium investment account.

SAFE probability : 0.22%
FRAUD probability: 99.78%
PREDICTION       : FRAUD

Message:
You can double your money in just seven days. Pay the processing fee now to recei

In [ ]:
import pandas as pd

spam_df = pd.read_csv(
    "/content/drive/MyDrive/ElderGuard/data/spam.csv",
    encoding="latin-1"
)

print("Kaggle spam.csv loaded successfully!")
print("Rows:", len(spam_df))
print("Columns:", spam_df.columns.tolist())

Kaggle spam.csv loaded successfully!
Rows: 5572
Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


In [ ]:
# Check which test messages came from the Kaggle SMS dataset

# v2 contains the actual SMS messages
kaggle_texts = set(
    spam_df["v2"].astype(str).str.strip()
)

print("Kaggle messages:", len(kaggle_texts))

# Find Kaggle messages present in our test set
kaggle_test = test_df[
    test_df["text"].astype(str).str.strip().isin(kaggle_texts)
].copy()

print("Kaggle messages found in test set:", len(kaggle_test))

Kaggle messages: 5158
Kaggle messages found in test set: 1022


In [ ]:
# Add predictions to the Kaggle test messages

kaggle_test = kaggle_test.reset_index(drop=True)

kaggle_test["prediction"] = [
    all_predictions[i]
    for i in kaggle_test.index
]

# Find incorrect predictions
kaggle_misses = kaggle_test[
    kaggle_test["label"] != kaggle_test["prediction"]
]

print("\n" + "=" * 60)
print("KAGGLE SPAM.CSV EVALUATION")
print("=" * 60)

print("\nKaggle messages in test set:", len(kaggle_test))
print("Incorrect predictions:", len(kaggle_misses))
print("Correct predictions:", len(kaggle_test) - len(kaggle_misses))

accuracy = (
    (kaggle_test["label"] == kaggle_test["prediction"]).mean()
    * 100
)

print(f"Kaggle test accuracy: {accuracy:.2f}%")

print("\n" + "=" * 60)
print("KAGGLE MISSES")
print("=" * 60)

for _, row in kaggle_misses.iterrows():
    print("\nMessage:", row["text"])
    print("Actual:", "FRAUD" if row["label"] == 1 else "SAFE")
    print("Predicted:", "FRAUD" if row["prediction"] == 1 else "SAFE")


KAGGLE SPAM.CSV EVALUATION

Kaggle messages in test set: 1022
Incorrect predictions: 256
Correct predictions: 766
Kaggle test accuracy: 74.95%

KAGGLE MISSES

Message: 83039 62735=å£450 UK Break AccommodationVouchers terms & conditions apply. 2 claim you mustprovide your claim number which is 15541
Actual: FRAUD
Predicted: SAFE

Message: K do I need a login or anything
Actual: SAFE
Predicted: FRAUD

Message: That way transport is less problematic than on sat night. By the way, if u want to ask  n  to join my bday, feel free. But need to know definite nos as booking on fri.
Actual: SAFE
Predicted: FRAUD

Message: You are being contacted by our dating service by someone you know! To find out who it is, call from a land line 09050000878. PoBox45W2TG150P
Actual: FRAUD
Predicted: SAFE

Message: You have won a guaranteed å£200 award or even å£1000 cashto claim UR award call free on 08000407165 (18+) 2 stop getstop on 88222 PHP
Actual: FRAUD
Predicted: SAFE

Message: U horrible gal... U knew

In [ ]:
print("=" * 70)
print("KAGGLE MISS ANALYSIS")
print("=" * 70)

print("\nTotal Kaggle misses:", len(kaggle_misses))

if len(kaggle_misses) == 0:
    print("\n🎉 No Kaggle messages were misclassified!")
else:
    for i, (_, row) in enumerate(kaggle_misses.iterrows(), start=1):
        print("\n" + "-" * 70)
        print(f"MISS {i}")
        print("-" * 70)

        print("Message:")
        print(row["text"])

        print("\nActual:",
              "FRAUD" if row["label"] == 1 else "SAFE")

        print("Predicted:",
              "FRAUD" if row["prediction"] == 1 else "SAFE")

KAGGLE MISS ANALYSIS

Total Kaggle misses: 256

----------------------------------------------------------------------
MISS 1
----------------------------------------------------------------------
Message:
83039 62735=å£450 UK Break AccommodationVouchers terms & conditions apply. 2 claim you mustprovide your claim number which is 15541

Actual: FRAUD
Predicted: SAFE

----------------------------------------------------------------------
MISS 2
----------------------------------------------------------------------
Message:
K do I need a login or anything

Actual: SAFE
Predicted: FRAUD

----------------------------------------------------------------------
MISS 3
----------------------------------------------------------------------
Message:
That way transport is less problematic than on sat night. By the way, if u want to ask  n  to join my bday, feel free. But need to know definite nos as booking on fri.

Actual: SAFE
Predicted: FRAUD

--------------------------------------------------

In [ ]:
print("=" * 80)
print("ELDERGUARD-SPECIFIC ERROR CHECK")
print("=" * 80)

elderguard_categories = {
    "Digital Arrest": [
        "This is CBI calling. Your Aadhaar is linked to a money laundering case. You are under digital arrest. Do not disconnect.",
        "Your SIM has been involved in an illegal case. Stay on video call and transfer money to the RBI verification account.",
        "I am calling from the police department. Do not tell your family about this case."
    ],

    "Investment Scam": [
        "Our trading experts guarantee 100% returns. Transfer ₹50000 today to activate your account.",
        "You can double your money in seven days. Pay the processing fee now to receive your profit.",
        "This is a guaranteed investment opportunity. Send ₹25000 immediately."
    ],

    "KYC Scam": [
        "Your bank account will be frozen unless you complete KYC immediately. Share your OTP.",
        "Your KYC has expired. Send your Aadhaar number and bank details immediately.",
        "Complete KYC verification now or your pension account will be suspended."
    ],

    "Remote Access Scam": [
        "Install AnyDesk so our bank officer can access your screen and verify your account.",
        "Download this remote access application immediately to fix your banking problem.",
        "Install TeamViewer and share your screen so we can verify the transaction."
    ],

    "OTP Scam": [
        "Your bank account has a suspicious transaction. Tell me the OTP you received.",
        "Provide your ATM PIN and OTP to complete the verification.",
        "Send your card number, PIN and password immediately."
    ],

    "Payment Pressure": [
        "Transfer the money immediately to the verification account or legal action will be taken.",
        "Pay the processing fee now or your bank account will be frozen.",
        "Send ₹10000 immediately to release your blocked account."
    ],

    "Safe Banking": [
        "Your monthly bank statement is available. Please check it through your usual banking application.",
        "Your salary has been credited to your bank account. Check it in your banking app.",
        "Your bank branch will be closed on Sunday. Normal services resume Monday."
    ],

    "Safe Bills": [
        "Your electricity bill is due tomorrow. Please pay through your normal electricity provider.",
        "Your internet bill is due this week. You can pay using your usual provider.",
        "Your water bill has been generated. Check the amount through the official website."
    ]
}

elderguard_summary = []

for category, messages in elderguard_categories.items():

    correct = 0

    print("\n" + "=" * 70)
    print(category)
    print("=" * 70)

    for message in messages:

        inputs = saved_tokenizer(
            message,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = saved_model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=1)

        safe_probability = probabilities[0][0].item()
        fraud_probability = probabilities[0][1].item()

        prediction = 1 if fraud_probability >= safe_probability else 0

        # First 6 categories are FRAUD
        actual = 1 if category not in ["Safe Banking", "Safe Bills"] else 0

        if prediction == actual:
            correct += 1

        print("\nMessage:", message)
        print(
            f"SAFE: {safe_probability*100:.2f}% | "
            f"FRAUD: {fraud_probability*100:.2f}%"
        )
        print(
            "Prediction:",
            "FRAUD" if prediction == 1 else "SAFE"
        )
        print(
            "Correct:",
            "YES" if prediction == actual else "NO"
        )

    accuracy = correct / len(messages) * 100

    elderguard_summary.append({
        "category": category,
        "correct": correct,
        "total": len(messages),
        "accuracy": accuracy
    })

print("\n" + "=" * 80)
print("ELDERGUARD CATEGORY SUMMARY")
print("=" * 80)

for row in elderguard_summary:
    print(
        f"{row['category']:25} "
        f"{row['correct']}/{row['total']} "
        f"({row['accuracy']:.2f}%)"
    )

ELDERGUARD-SPECIFIC ERROR CHECK

Digital Arrest

Message: This is CBI calling. Your Aadhaar is linked to a money laundering case. You are under digital arrest. Do not disconnect.
SAFE: 1.19% | FRAUD: 98.81%
Prediction: FRAUD
Correct: YES

Message: Your SIM has been involved in an illegal case. Stay on video call and transfer money to the RBI verification account.
SAFE: 0.29% | FRAUD: 99.71%
Prediction: FRAUD
Correct: YES

Message: I am calling from the police department. Do not tell your family about this case.
SAFE: 29.30% | FRAUD: 70.70%
Prediction: FRAUD
Correct: YES

Investment Scam

Message: Our trading experts guarantee 100% returns. Transfer ₹50000 today to activate your account.
SAFE: 0.25% | FRAUD: 99.75%
Prediction: FRAUD
Correct: YES

Message: You can double your money in seven days. Pay the processing fee now to receive your profit.
SAFE: 1.49% | FRAUD: 98.51%
Prediction: FRAUD
Correct: YES

Message: This is a guaranteed investment opportunity. Send ₹25000 immediately.
SAFE

In [ ]:
import pandas as pd

# Load the combined dataset
combined_df = pd.read_csv(
    "/content/drive/MyDrive/ElderGuard/data/elderguard_combined.csv"
)

print("Columns:", combined_df.columns.tolist())
print("Total rows:", len(combined_df))

# Search for ElderGuard remote-access terms
remote_keywords = [
    "anydesk",
    "teamviewer",
    "remote access",
    "screen sharing",
    "screen share",
    "remote application",
    "install the app",
    "download the app",
    "apk"
]

pattern = "|".join(remote_keywords)

remote_examples = combined_df[
    combined_df["text"]
    .astype(str)
    .str.lower()
    .str.contains(pattern, regex=True, na=False)
].copy()

print("\n" + "=" * 70)
print("REMOTE ACCESS EXAMPLES IN CURRENT DATASET")
print("=" * 70)

print("\nNumber of matching messages:", len(remote_examples))

for _, row in remote_examples.head(30).iterrows():
    print("\nLabel:", "FRAUD" if row["label"] == 1 else "SAFE")
    print("Message:", row["text"])

Columns: ['text', 'label']
Total rows: 5980

REMOTE ACCESS EXAMPLES IN CURRENT DATASET

Number of matching messages: 17

Label: FRAUD
Message: Download this remote access application to fix your bank account

Label: FRAUD
Message: [Greetings], this is [Name], and we are from [Company] . We are calling you to dive into the festive spirit of Deepavali with our exclusive offers! Visit our company’s social media page [Company]  at [Link] page and click the link to place your order, adding radiance to your celebrations. Upon clicking, you will be guided to our secure messaging platform on Phone , our company’s social media page [Company]  at [Link], or Instagram, where our dedicated sellers stand ready to assist. To ensure a hassle-free ordering experience, consider downloading our specially crafted [APK FILE] accessible through this [Link]. Install now to amplify your access to our delightful offerings. Following the installation, confirm your order by providing your banking details for pa

In [ ]:
# STEP 31
# Create targeted ElderGuard remote-access fraud examples

remote_training_examples = [
    ("Install AnyDesk so the bank officer can access your phone and verify your account.", 1),
    ("Please download AnyDesk and share your screen to complete your bank verification.", 1),
    ("Install TeamViewer immediately so our banking support team can fix your account.", 1),
    ("Our bank officer needs remote access to your computer to verify the suspicious transaction.", 1),
    ("Download this remote access app and allow us to control your screen for KYC verification.", 1),
    ("Share your screen through AnyDesk so we can verify your bank account.", 1),
    ("Install TeamViewer and give our customer support agent access to your device.", 1),
    ("Your bank account has a security problem. Install this remote access application immediately.", 1),
    ("To remove the virus from your banking application, install AnyDesk and give us remote access.", 1),
    ("Our technical support team needs remote access to your device. Install AnyDesk now.", 1),
    ("Download the remote support application and share your screen with the bank representative.", 1),
    ("For KYC verification, install TeamViewer and allow the officer to control your screen.", 1),
    ("Your account will remain blocked unless you install this remote access application.", 1),
    ("Open AnyDesk and provide the access code so we can verify your banking transaction.", 1),
    ("Install a remote support app and give us access to your phone to complete the refund.", 1),
    ("The bank requires remote screen access to investigate your suspicious transaction.", 1),
    ("Please install AnyDesk and share the access code with our bank verification officer.", 1),
    ("Download TeamViewer now so we can remotely fix your blocked bank account.", 1),
    ("Our customer-care agent needs screen sharing access to complete your KYC verification.", 1),
    ("Install this APK and enable remote access so we can resolve your banking issue.", 1)
]

remote_training_df = pd.DataFrame(
    remote_training_examples,
    columns=["text", "label"]
)

print("=" * 70)
print("TARGETED REMOTE-ACCESS TRAINING DATA")
print("=" * 70)

print("New examples:", len(remote_training_df))
print("\nLabel distribution:")
print(remote_training_df["label"].value_counts())

display(remote_training_df.head())

TARGETED REMOTE-ACCESS TRAINING DATA
New examples: 20

Label distribution:
label
1    20
Name: count, dtype: int64


,text,label
0,Install AnyDesk so the bank officer can access...,1
1,Please download AnyDesk and share your screen ...,1
2,Install TeamViewer immediately so our banking ...,1
3,Our bank officer needs remote access to your c...,1
4,Download this remote access app and allow us t...,1


In [ ]:
# STEP 32
# Create safe remote-access examples

safe_remote_examples = [
    ("I used AnyDesk to help my brother fix his computer after he asked me for help.", 0),
    ("Our IT department scheduled a remote support session for my laptop tomorrow.", 0),
    ("The company technician will use TeamViewer during the support appointment I requested.", 0),
    ("I contacted the official software support team and scheduled a remote assistance session.", 0),
    ("The computer technician is helping me remotely after I booked a support appointment.", 0),
    ("Our office IT team uses remote access to maintain company computers.", 0),
    ("I requested remote support from the official service provider to fix my computer.", 0),
    ("The technician will connect remotely after I confirm my scheduled support appointment.", 0),
    ("Our company's IT department provides remote screen support for employees.", 0),
    ("I contacted customer support through the official website and requested remote assistance.", 0)
]

safe_remote_df = pd.DataFrame(
    safe_remote_examples,
    columns=["text", "label"]
)

print("=" * 70)
print("SAFE REMOTE-ACCESS TRAINING DATA")
print("=" * 70)

print("New SAFE examples:", len(safe_remote_df))
print("\nLabel distribution:")
print(safe_remote_df["label"].value_counts())

display(safe_remote_df)

SAFE REMOTE-ACCESS TRAINING DATA
New SAFE examples: 10

Label distribution:
label
0    10
Name: count, dtype: int64


,text,label
0,I used AnyDesk to help my brother fix his comp...,0
1,Our IT department scheduled a remote support s...,0
2,The company technician will use TeamViewer dur...,0
3,I contacted the official software support team...,0
4,The computer technician is helping me remotely...,0
5,Our office IT team uses remote access to maint...,0
6,I requested remote support from the official s...,0
7,The technician will connect remotely after I c...,0
8,Our company's IT department provides remote sc...,0
9,I contacted customer support through the offic...,0


In [ ]:
# Combine original training data with targeted remote-access examples

train_augmented = pd.concat(
    [train_df, remote_training_df, safe_remote_df],
    ignore_index=True
)

print("=" * 70)
print("AUGMENTED TRAINING DATA")
print("=" * 70)

print("\nOriginal training examples:", len(train_df))
print("New remote-access FRAUD examples:", len(remote_training_df))
print("New remote-access SAFE examples:", len(safe_remote_df))
print("Total augmented training examples:", len(train_augmented))

print("\nLabel distribution:")
print(train_augmented["label"].value_counts())

print("\nLast 10 examples:")
display(train_augmented.tail(10))

AUGMENTED TRAINING DATA

Original training examples: 4784
New remote-access FRAUD examples: 20
New remote-access SAFE examples: 10
Total augmented training examples: 4814

Label distribution:
label
0    3951
1     863
Name: count, dtype: int64

Last 10 examples:


,text,label
4804,I used AnyDesk to help my brother fix his comp...,0
4805,Our IT department scheduled a remote support s...,0
4806,The company technician will use TeamViewer dur...,0
4807,I contacted the official software support team...,0
4808,The computer technician is helping me remotely...,0
4809,Our office IT team uses remote access to maint...,0
4810,I requested remote support from the official s...,0
4811,The technician will connect remotely after I c...,0
4812,Our company's IT department provides remote sc...,0
4813,I contacted customer support through the offic...,0


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Tokenize the augmented training data
train_encodings = saved_tokenizer(
    train_augmented["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# Convert labels to PyTorch tensor
train_labels = torch.tensor(
    train_augmented["label"].tolist(),
    dtype=torch.long
)

# Create PyTorch dataset
train_dataset_augmented = TensorDataset(
    train_encodings["input_ids"],
    train_encodings["attention_mask"],
    train_labels
)

# Create DataLoader
train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=32,
    shuffle=True
)

print("=" * 70)
print("AUGMENTED TRAINING DATALOADER")
print("=" * 70)

print("Training examples:", len(train_dataset_augmented))
print("Batch size:", 32)
print("Number of batches:", len(train_loader_augmented))

print("\nDataLoader created successfully!")

AUGMENTED TRAINING DATALOADER
Training examples: 4814
Batch size: 32
Number of batches: 151

DataLoader created successfully!


In [ ]:
from transformers import AutoModelForSequenceClassification

# Create a fresh DistilBERT model
augmented_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Move model to GPU/CPU
augmented_model = augmented_model.to(device)

print("=" * 70)
print("FRESH ELDERGUARD MODEL")
print("=" * 70)
print("Model: DistilBERT")
print("Number of labels: 2")
print("Device:", device)
print("\nFresh model created successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FRESH ELDERGUARD MODEL
Model: DistilBERT
Number of labels: 2
Device: cuda

Fresh model created successfully!


In [ ]:
import torch
from torch.optim import AdamW

# Optimizer
optimizer = AdamW(
    augmented_model.parameters(),
    lr=2e-5
)

# Number of epochs
epochs = 3

print("=" * 70)
print("TRAINING AUGMENTED ELDERGUARD MODEL")
print("=" * 70)

for epoch in range(epochs):

    augmented_model.train()

    total_loss = 0

    for batch in train_loader_augmented:

        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = augmented_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update model
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader_augmented)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Average Loss: {average_loss:.4f}"
    )

print("\nTraining completed successfully!")

TRAINING AUGMENTED ELDERGUARD MODEL
Epoch 1/3 | Average Loss: 0.1541
Epoch 2/3 | Average Loss: 0.0345
Epoch 3/3 | Average Loss: 0.0129

Training completed successfully!


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch

augmented_model.eval()

all_predictions = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = augmented_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

# Get the original test labels
all_labels = test_df["label"].tolist()

# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print("=" * 70)
print("AUGMENTED ELDERGUARD MODEL — EVALUATION")
print("=" * 70)

print(f"Test samples : {len(all_labels)}")
print(f"Accuracy     : {accuracy:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 Score     : {f1:.4f}")

AUGMENTED ELDERGUARD MODEL — EVALUATION
Test samples : 1196
Accuracy     : 0.9950
Precision    : 0.9904
Recall       : 0.9810
F1 Score     : 0.9857


In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(all_labels, all_predictions)

print("=" * 70)
print("AUGMENTED MODEL — CONFUSION MATRIX")
print("=" * 70)

print("\n                 Predicted")
print("              SAFE    FRAUD")
print(f"Actual SAFE   {cm[0][0]:4d}    {cm[0][1]:4d}")
print(f"Actual FRAUD  {cm[1][0]:4d}    {cm[1][1]:4d}")

print("\nFalse Positives:", cm[0][1])
print("False Negatives:", cm[1][0])

AUGMENTED MODEL — CONFUSION MATRIX

                 Predicted
              SAFE    FRAUD
Actual SAFE    983       2
Actual FRAUD     4     207

False Positives: 2
False Negatives: 4


In [ ]:
remote_test_messages = {

    "Remote Access FRAUD 1":
        "The bank says I must install AnyDesk and give the agent access to my phone "
        "before they can investigate the suspicious payment.",

    "Remote Access FRAUD 2":
        "A customer service representative asked me to open TeamViewer and share "
        "the access code to resolve my blocked account.",

    "Remote Access FRAUD 3":
        "To complete the refund, the bank officer wants me to download a remote "
        "support app and let them control my screen.",

    "Safe Remote Support 1":
        "I scheduled a remote computer repair with the technician from the official "
        "service company after contacting them myself.",

    "Safe Remote Support 2":
        "My workplace IT team will use TeamViewer during the remote support session "
        "that I requested for my office computer.",

    "Safe Remote Support 3":
        "I asked the official software support team for remote assistance because "
        "my computer was not working properly."
}

print("=" * 70)
print("STEP 39 — REMOTE-ACCESS TEST")
print("=" * 70)

for category, message in remote_test_messages.items():

    # Original model
    inputs = saved_tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        old_output = saved_model(**inputs)
        old_probs = torch.softmax(old_output.logits, dim=1)

    old_prediction = torch.argmax(old_probs, dim=1).item()
    old_fraud = old_probs[0][1].item() * 100

    # Augmented model
    with torch.no_grad():
        new_output = augmented_model(**inputs)
        new_probs = torch.softmax(new_output.logits, dim=1)

    new_prediction = torch.argmax(new_probs, dim=1).item()
    new_fraud = new_probs[0][1].item() * 100

    old_label = "FRAUD" if old_prediction == 1 else "SAFE"
    new_label = "FRAUD" if new_prediction == 1 else "SAFE"

    print("\n" + "-" * 70)
    print(category)
    print("-" * 70)
    print(message)

    print(f"\nOriginal model : {old_label} ({old_fraud:.2f}% FRAUD)")
    print(f"Augmented model: {new_label} ({new_fraud:.2f}% FRAUD)")

STEP 39 — REMOTE-ACCESS TEST

----------------------------------------------------------------------
Remote Access FRAUD 1
----------------------------------------------------------------------
The bank says I must install AnyDesk and give the agent access to my phone before they can investigate the suspicious payment.

Original model : FRAUD (83.05% FRAUD)
Augmented model: FRAUD (94.55% FRAUD)

----------------------------------------------------------------------
Remote Access FRAUD 2
----------------------------------------------------------------------
A customer service representative asked me to open TeamViewer and share the access code to resolve my blocked account.

Original model : FRAUD (97.05% FRAUD)
Augmented model: FRAUD (75.66% FRAUD)

----------------------------------------------------------------------
Remote Access FRAUD 3
----------------------------------------------------------------------
To complete the refund, the bank officer wants me to download a remote suppo

In [ ]:
remote_failure_tests = [
    "Install AnyDesk so our bank officer can access your screen and verify your account.",

    "Download this remote access application immediately so we can fix your banking problem.",

    "Please install TeamViewer and share your screen so our support team can verify the transaction."
]

print("=" * 70)
print("STEP 40 — PREVIOUS REMOTE-ACCESS FAILURES")
print("=" * 70)

correct = 0

for i, message in enumerate(remote_failure_tests, start=1):

    inputs = saved_tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = augmented_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)

    prediction = torch.argmax(probabilities, dim=1).item()
    fraud_probability = probabilities[0][1].item() * 100

    label = "FRAUD" if prediction == 1 else "SAFE"

    if label == "FRAUD":
        correct += 1

    print("\n" + "-" * 70)
    print(f"Test {i}")
    print("-" * 70)
    print(message)
    print(f"\nPrediction: {label}")
    print(f"FRAUD probability: {fraud_probability:.2f}%")

print("\n" + "=" * 70)
print(f"RESULT: {correct}/3 correctly detected as FRAUD")
print("=" * 70)

STEP 40 — PREVIOUS REMOTE-ACCESS FAILURES

----------------------------------------------------------------------
Test 1
----------------------------------------------------------------------
Install AnyDesk so our bank officer can access your screen and verify your account.

Prediction: FRAUD
FRAUD probability: 98.72%

----------------------------------------------------------------------
Test 2
----------------------------------------------------------------------
Download this remote access application immediately so we can fix your banking problem.

Prediction: FRAUD
FRAUD probability: 78.72%

----------------------------------------------------------------------
Test 3
----------------------------------------------------------------------
Please install TeamViewer and share your screen so our support team can verify the transaction.

Prediction: FRAUD
FRAUD probability: 91.05%

RESULT: 3/3 correctly detected as FRAUD


In [ ]:
remote_failure_tests = [
    "Install AnyDesk so our bank officer can access your screen and verify your account.",

    "Download this remote access application immediately so we can fix your banking problem.",

    "Please install TeamViewer and share your screen so our support team can verify the transaction."
]

print("=" * 70)
print("STEP 40 — PREVIOUS REMOTE-ACCESS FAILURES")
print("=" * 70)

correct = 0

for i, message in enumerate(remote_failure_tests, start=1):

    inputs = saved_tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = augmented_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)

    prediction = torch.argmax(probabilities, dim=1).item()
    fraud_probability = probabilities[0][1].item() * 100

    label = "FRAUD" if prediction == 1 else "SAFE"

    if label == "FRAUD":
        correct += 1

    print("\n" + "-" * 70)
    print(f"Test {i}")
    print("-" * 70)
    print(message)
    print(f"\nPrediction: {label}")
    print(f"FRAUD probability: {fraud_probability:.2f}%")

print("\n" + "=" * 70)
print(f"RESULT: {correct}/3 correctly detected as FRAUD")
print("=" * 70)

STEP 40 — PREVIOUS REMOTE-ACCESS FAILURES

----------------------------------------------------------------------
Test 1
----------------------------------------------------------------------
Install AnyDesk so our bank officer can access your screen and verify your account.

Prediction: FRAUD
FRAUD probability: 98.72%

----------------------------------------------------------------------
Test 2
----------------------------------------------------------------------
Download this remote access application immediately so we can fix your banking problem.

Prediction: FRAUD
FRAUD probability: 78.72%

----------------------------------------------------------------------
Test 3
----------------------------------------------------------------------
Please install TeamViewer and share your screen so our support team can verify the transaction.

Prediction: FRAUD
FRAUD probability: 91.05%

RESULT: 3/3 correctly detected as FRAUD


In [ ]:
print("=" * 70)
print("STEP 41 — AUGMENTED MODEL ELDERGUARD TEST")
print("=" * 70)

category_results = {}

for category, messages in elderguard_test_messages.items():

    correct = 0
    total = len(messages)

    print("\n" + "=" * 70)
    print(category)
    print("=" * 70)

    for i, message in enumerate(messages, start=1):

        inputs = saved_tokenizer(
            message,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = augmented_model(**inputs)
            probabilities = torch.softmax(outputs.logits, dim=1)

        prediction = torch.argmax(probabilities, dim=1).item()
        fraud_probability = probabilities[0][1].item() * 100

        predicted_label = "FRAUD" if prediction == 1 else "SAFE"

        # Expected label based on category
        expected_label = (
            "FRAUD"
            if category in [
                "Digital Arrest",
                "Investment Scam",
                "KYC Scam",
                "Remote Access Scam",
                "OTP Credential Scam",
                "Payment Pressure"
            ]
            else "SAFE"
        )

        is_correct = predicted_label == expected_label

        if is_correct:
            correct += 1

        status = "✓" if is_correct else "✗"

        print(
            f"{status} Test {i}: "
            f"Expected={expected_label}, "
            f"Predicted={predicted_label}, "
            f"FRAUD={fraud_probability:.2f}%"
        )

    accuracy = correct / total * 100
    category_results[category] = (correct, total, accuracy)

    print(f"\nCategory result: {correct}/{total} ({accuracy:.2f}%)")


print("\n" + "=" * 70)
print("ELDERGUARD CATEGORY SUMMARY")
print("=" * 70)

total_correct = 0
total_tests = 0

for category, (correct, total, accuracy) in category_results.items():

    print(
        f"{category:<25} "
        f"{correct}/{total} "
        f"({accuracy:.2f}%)"
    )

    total_correct += correct
    total_tests += total

print("\n" + "=" * 70)
print(
    f"OVERALL: {total_correct}/{total_tests} "
    f"({total_correct / total_tests * 100:.2f}%)"
)
print("=" * 70)

STEP 41 — AUGMENTED MODEL ELDERGUARD TEST

Digital Arrest
✓ Test 1: Expected=FRAUD, Predicted=FRAUD, FRAUD=99.47%
✓ Test 2: Expected=FRAUD, Predicted=FRAUD, FRAUD=99.73%
✓ Test 3: Expected=FRAUD, Predicted=FRAUD, FRAUD=94.05%

Category result: 3/3 (100.00%)

Investment Scam
✓ Test 1: Expected=FRAUD, Predicted=FRAUD, FRAUD=99.77%
✓ Test 2: Expected=FRAUD, Predicted=FRAUD, FRAUD=99.55%
✓ Test 3: Expected=FRAUD, Predicted=FRAUD, FRAUD=99.68%

Category result: 3/3 (100.00%)

KYC Scam
✓ Test 1: Expected=FRAUD, Predicted=FRAUD, FRAUD=98.75%
✓ Test 2: Expected=FRAUD, Predicted=FRAUD, FRAUD=95.69%
✓ Test 3: Expected=FRAUD, Predicted=FRAUD, FRAUD=94.45%

Category result: 3/3 (100.00%)

Remote Access Scam
✓ Test 1: Expected=FRAUD, Predicted=FRAUD, FRAUD=98.72%
✓ Test 2: Expected=FRAUD, Predicted=FRAUD, FRAUD=78.72%
✓ Test 3: Expected=FRAUD, Predicted=FRAUD, FRAUD=91.05%

Category result: 3/3 (100.00%)

OTP Credential Scam
✓ Test 1: Expected=FRAUD, Predicted=FRAUD, FRAUD=64.31%
✓ Test 2: Expected

In [ ]:
safe_banking_examples = [
    ("I checked my bank balance using my usual banking application this morning.", 0),
    ("My monthly bank statement is available in the official banking app.", 0),
    ("My salary was credited to my account today and I checked the transaction.", 0),
    ("I received a notification that my fixed deposit interest was credited.", 0),
    ("I visited my bank branch to update my address and completed the process.", 0),
    ("The bank sent me my monthly account statement through the official app.", 0),
    ("I transferred money to my daughter using my normal banking application.", 0),
    ("I checked the details of my recent transactions in my bank account.", 0),
    ("The bank notified me that my scheduled EMI payment was completed successfully.", 0),
    ("I received my bank statement and reviewed my transactions for the month.", 0)
]

safe_banking_df = pd.DataFrame(
    safe_banking_examples,
    columns=["text", "label"]
)

print("=" * 70)
print("SAFE BANKING TRAINING DATA")
print("=" * 70)

print("New SAFE banking examples:", len(safe_banking_df))

print("\nLabel distribution:")
print(safe_banking_df["label"].value_counts())

display(safe_banking_df)

SAFE BANKING TRAINING DATA
New SAFE banking examples: 10

Label distribution:
label
0    10
Name: count, dtype: int64


,text,label
0,I checked my bank balance using my usual banki...,0
1,My monthly bank statement is available in the ...,0
2,My salary was credited to my account today and...,0
3,I received a notification that my fixed deposi...,0
4,I visited my bank branch to update my address ...,0
5,The bank sent me my monthly account statement ...,0
6,I transferred money to my daughter using my no...,0
7,I checked the details of my recent transaction...,0
8,The bank notified me that my scheduled EMI pay...,0
9,I received my bank statement and reviewed my t...,0


In [ ]:
# Add SAFE banking examples to the current augmented training data

train_augmented_v2 = pd.concat(
    [
        train_augmented,
        safe_banking_df
    ],
    ignore_index=True
)

print("=" * 70)
print("ELDERGUARD TRAINING DATA V2")
print("=" * 70)

print("\nPrevious augmented training examples:", len(train_augmented))
print("New SAFE banking examples:", len(safe_banking_df))
print("Total training examples:", len(train_augmented_v2))

print("\nLabel distribution:")
print(train_augmented_v2["label"].value_counts())

print("\nLast 10 examples:")
display(train_augmented_v2.tail(10))

ELDERGUARD TRAINING DATA V2

Previous augmented training examples: 4814
New SAFE banking examples: 10
Total training examples: 4824

Label distribution:
label
0    3961
1     863
Name: count, dtype: int64

Last 10 examples:


,text,label
4814,I checked my bank balance using my usual banki...,0
4815,My monthly bank statement is available in the ...,0
4816,My salary was credited to my account today and...,0
4817,I received a notification that my fixed deposi...,0
4818,I visited my bank branch to update my address ...,0
4819,The bank sent me my monthly account statement ...,0
4820,I transferred money to my daughter using my no...,0
4821,I checked the details of my recent transaction...,0
4822,The bank notified me that my scheduled EMI pay...,0
4823,I received my bank statement and reviewed my t...,0


In [ ]:
# Tokenize V2 training data

train_encodings_v2 = saved_tokenizer(
    train_augmented_v2["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# Convert labels to tensors

train_labels_v2 = torch.tensor(
    train_augmented_v2["label"].tolist(),
    dtype=torch.long
)

# Create PyTorch dataset

train_dataset_v2 = TensorDataset(
    train_encodings_v2["input_ids"],
    train_encodings_v2["attention_mask"],
    train_labels_v2
)

# Create DataLoader

train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=32,
    shuffle=True
)

print("=" * 70)
print("ELDERGUARD V2 DATALOADER")
print("=" * 70)

print("Training examples:", len(train_dataset_v2))
print("Batch size:", 32)
print("Number of batches:", len(train_loader_v2))

print("\nV2 DataLoader created successfully!")

ELDERGUARD V2 DATALOADER
Training examples: 4824
Batch size: 32
Number of batches: 151

V2 DataLoader created successfully!


In [ ]:
# Tokenize V2 training data

train_encodings_v2 = saved_tokenizer(
    train_augmented_v2["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# Convert labels to tensors

train_labels_v2 = torch.tensor(
    train_augmented_v2["label"].tolist(),
    dtype=torch.long
)

# Create PyTorch dataset

train_dataset_v2 = TensorDataset(
    train_encodings_v2["input_ids"],
    train_encodings_v2["attention_mask"],
    train_labels_v2
)

# Create DataLoader

train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=32,
    shuffle=True
)

print("=" * 70)
print("ELDERGUARD V2 DATALOADER")
print("=" * 70)

print("Training examples:", len(train_dataset_v2))
print("Batch size:", 32)
print("Number of batches:", len(train_loader_v2))

print("\nV2 DataLoader created successfully!")

ELDERGUARD V2 DATALOADER
Training examples: 4824
Batch size: 32
Number of batches: 151

V2 DataLoader created successfully!


In [ ]:
from transformers import AutoModelForSequenceClassification

# Create a fresh V2 DistilBERT model
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Move model to GPU
model_v2 = model_v2.to(device)

print("=" * 70)
print("ELDERGUARD V2 MODEL")
print("=" * 70)

print("Model: DistilBERT")
print("Number of labels: 2")
print("Device:", device)

print("\nFresh V2 model created successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ELDERGUARD V2 MODEL
Model: DistilBERT
Number of labels: 2
Device: cuda

Fresh V2 model created successfully!


In [ ]:
from torch.optim import AdamW

# Create optimizer
optimizer_v2 = AdamW(
    model_v2.parameters(),
    lr=2e-5
)

# Number of epochs
epochs = 3

print("=" * 70)
print("TRAINING ELDERGUARD V2")
print("=" * 70)

for epoch in range(epochs):

    model_v2.train()

    total_loss = 0

    for batch in train_loader_v2:

        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        # Clear previous gradients
        optimizer_v2.zero_grad()

        # Forward pass
        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer_v2.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader_v2)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Average Loss: {average_loss:.4f}"
    )

print("\nV2 training completed successfully!")

TRAINING ELDERGUARD V2
Epoch 1/3 | Average Loss: 0.1365
Epoch 2/3 | Average Loss: 0.0292
Epoch 3/3 | Average Loss: 0.0119

V2 training completed successfully!


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_v2.eval()

v2_predictions = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        v2_predictions.extend(
            predictions.cpu().numpy()
        )

# Original untouched test labels
v2_labels = test_df["label"].tolist()

# Calculate metrics
v2_accuracy = accuracy_score(
    v2_labels,
    v2_predictions
)

v2_precision = precision_score(
    v2_labels,
    v2_predictions
)

v2_recall = recall_score(
    v2_labels,
    v2_predictions
)

v2_f1 = f1_score(
    v2_labels,
    v2_predictions
)

print("=" * 70)
print("ELDERGUARD V2 — EVALUATION")
print("=" * 70)

print(f"Test samples : {len(v2_labels)}")
print(f"Accuracy     : {v2_accuracy:.4f}")
print(f"Precision    : {v2_precision:.4f}")
print(f"Recall       : {v2_recall:.4f}")
print(f"F1 Score     : {v2_f1:.4f}")

ELDERGUARD V2 — EVALUATION
Test samples : 1196
Accuracy     : 0.9925
Precision    : 0.9903
Recall       : 0.9668
F1 Score     : 0.9784


In [ ]:
print("=" * 75)
print("ELDERGUARD MODEL COMPARISON")
print("=" * 75)

print("\n                 Accuracy    Precision    Recall      F1")
print("-" * 75)

print(f"Original Model   {0.9933:.4f}      {0.9856:.4f}      {0.9763:.4f}      {0.9810:.4f}")
print(f"V1 Model         {0.9916:.4f}      {0.9718:.4f}      {0.9810:.4f}      {0.9764:.4f}")
print(f"V2 Model         {v2_accuracy:.4f}      {v2_precision:.4f}      {v2_recall:.4f}      {v2_f1:.4f}")

ELDERGUARD MODEL COMPARISON

                 Accuracy    Precision    Recall      F1
---------------------------------------------------------------------------
Original Model   0.9933      0.9856      0.9763      0.9810
V1 Model         0.9916      0.9718      0.9810      0.9764
V2 Model         0.9925      0.9903      0.9668      0.9784


In [ ]:
from sklearn.metrics import confusion_matrix

v2_cm = confusion_matrix(v2_labels, v2_predictions)

print("=" * 60)
print("ELDERGUARD V2 — CONFUSION MATRIX")
print("=" * 60)

print("\n                 Predicted")
print("              SAFE    FRAUD")
print(f"Actual SAFE   {v2_cm[0][0]:5d}   {v2_cm[0][1]:5d}")
print(f"Actual FRAUD  {v2_cm[1][0]:5d}   {v2_cm[1][1]:5d}")

print("\nFalse Positives:", v2_cm[0][1])
print("False Negatives:", v2_cm[1][0])

ELDERGUARD V2 — CONFUSION MATRIX

                 Predicted
              SAFE    FRAUD
Actual SAFE     983       2
Actual FRAUD      7     204

False Positives: 2
False Negatives: 7


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("ELDERGUARD V2 — CLASSIFICATION REPORT")
print("=" * 60)

print("Accuracy:", accuracy_score(v2_labels, v2_predictions))
print("Precision:", precision_score(v2_labels, v2_predictions, zero_division=0))
print("Recall:", recall_score(v2_labels, v2_predictions, zero_division=0))
print("F1-score:", f1_score(v2_labels, v2_predictions, zero_division=0))

print("\nDetailed Classification Report:")
print(classification_report(
    v2_labels,
    v2_predictions,
    target_names=["SAFE", "FRAUD"],
    zero_division=0
))

ELDERGUARD V2 — CLASSIFICATION REPORT
Accuracy: 0.9924749163879598
Precision: 0.9902912621359223
Recall: 0.966824644549763
F1-score: 0.9784172661870504

Detailed Classification Report:
              precision    recall  f1-score   support

        SAFE       0.99      1.00      1.00       985
       FRAUD       0.99      0.97      0.98       211

    accuracy                           0.99      1196
   macro avg       0.99      0.98      0.99      1196
weighted avg       0.99      0.99      0.99      1196



In [ ]:
print("Original saved model:", "saved_model" in globals())
print("V1 augmented model:", "augmented_model" in globals())
print("V2 model:", "model_v2" in globals())
print("Tokenizer:", "saved_tokenizer" in globals())
print("V1 training data:", "train_augmented" in globals())
print("V2 training data:", "train_augmented_v2" in globals())

Original saved model: True
V1 augmented model: True
V2 model: True
Tokenizer: True
V1 training data: True
V2 training data: True


In [ ]:
import os

model_root = "/content/drive/MyDrive/ElderGuard/model"

if os.path.exists(model_root):
    print("Available model folders:")
    for name in sorted(os.listdir(model_root)):
        path = os.path.join(model_root, name)
        if os.path.isdir(path):
            print("-", name)
else:
    print("Model directory not found. Check whether Google Drive is mounted.")

Available model folders:
- elderguard_distilbert
- elderguard_distilbert_v1
- elderguard_distilbert_v2


In [ ]:
import os

v1_path = "/content/drive/MyDrive/ElderGuard/model/elderguard_distilbert_v1"

augmented_model.save_pretrained(v1_path)
saved_tokenizer.save_pretrained(v1_path)

print("V1 model saved:", os.path.exists(v1_path))
print("Saved files:", sorted(os.listdir(v1_path)))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

V1 model saved: True
Saved files: ['config.json', 'elderguard_model.onnx', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import os

v1_path = "/content/drive/MyDrive/ElderGuard/model/elderguard_distilbert_v1"

tokenizer_v1 = AutoTokenizer.from_pretrained(v1_path)
model_v1 = AutoModelForSequenceClassification.from_pretrained(v1_path)

model_v1.eval()

print("V1 tokenizer loaded:", tokenizer_v1 is not None)
print("V1 model loaded:", model_v1 is not None)
print("Number of labels:", model_v1.config.num_labels)
print("Label mapping:", model_v1.config.id2label)

test_text = "Your bank account will be blocked. Share your OTP immediately."
inputs = tokenizer_v1(
    test_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

with torch.no_grad():
    outputs = model_v1(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=1)[0]

predicted_class = torch.argmax(probabilities).item()

print("\nTest prediction:", predicted_class)
print("Class probabilities:", probabilities.tolist())

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

V1 tokenizer loaded: True
V1 model loaded: True
Number of labels: 2
Label mapping: {0: 'LABEL_0', 1: 'LABEL_1'}

Test prediction: 1
Class probabilities: [0.02135634422302246, 0.9786435961723328]


In [ ]:
# 1. Install required ONNX packages
!pip install -q onnx onnxruntime onnxscript

import os
import numpy as np
import onnxruntime as ort
import torch

v1_path = "/content/drive/MyDrive/ElderGuard/model/elderguard_distilbert_v1"
onnx_path = os.path.join(v1_path, "elderguard_model.onnx")
vocab_path = os.path.join(v1_path, "vocab.txt")

print("=" * 60)
print("FINAL STEP 56: SELF-CONTAINED ONNX & VOCAB EXPORT")
print("=" * 60)

# 2. Prepare sample input for tracing on CPU
model_v1_cpu = model_v1.to("cpu")
model_v1_cpu.eval()

sample_text = "Your bank account will be blocked. Share your OTP immediately."
inputs = tokenizer_v1(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=128,
)

# 3. Export single self-contained ONNX model (dynamo=False embeds weights for Android)
with torch.no_grad():
  torch.onnx.export(
      model_v1_cpu,
      (inputs["input_ids"], inputs["attention_mask"]),
      onnx_path,
      input_names=["input_ids", "attention_mask"],
      output_names=["logits"],
      dynamic_axes={
          "input_ids": {0: "batch_size"},
          "attention_mask": {0: "batch_size"},
          "logits": {0: "batch_size"},
      },
      opset_version=14,
      dynamo=False,
  )

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"\n✓ ONNX model exported: {onnx_path}")
print(f"✓ Model file size: {size_mb:.2f} MB (fully embedded weights)")

# 4. Export vocab.txt directly from tokenizer vocabulary
vocab = tokenizer_v1.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda item: item[1])
with open(vocab_path, "w", encoding="utf-8") as f:
  for token, _ in sorted_vocab:
    f.write(token + "\n")

print(f"✓ vocab.txt saved: {len(sorted_vocab)} tokens")

# 5. Final verification via ONNX Runtime
print("\n" + "=" * 60)
print("FINAL ONNX VERIFICATION")
print("=" * 60)

session = ort.InferenceSession(onnx_path)
onnx_inputs = {
    "input_ids": inputs["input_ids"].numpy(),
    "attention_mask": inputs["attention_mask"].numpy(),
}
onnx_logits = session.run(None, onnx_inputs)[0]


def softmax(x):
  e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
  return e_x / e_x.sum(axis=1, keepdims=True)


onnx_probs = softmax(onnx_logits)[0]
onnx_pred = int(np.argmax(onnx_probs))

print(
    "Prediction:",
    "FRAUD" if onnx_pred == 1 else "SAFE",
    f"({onnx_probs[1]*100:.2f}%)",
)
print("Files ready on Drive:")
for file in sorted(os.listdir(v1_path)):
  print(f" - {file}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 9.4 MB/s eta 0:00:00
FINAL STEP 56: SELF-CONTAINED ONNX & VOCAB EXPORT


/tmp/ipykernel_861/805395820.py:32: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:214: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/integrations/sdpa_attention.py:124: TracerWarning: Converting a tensor to a Python boolean 


✓ ONNX model exported: /content/drive/MyDrive/ElderGuard/model/elderguard_distilbert_v1/elderguard_model.onnx
✓ Model file size: 255.55 MB (fully embedded weights)
✓ vocab.txt saved: 30522 tokens

FINAL ONNX VERIFICATION
Prediction: FRAUD (97.86%)
Files ready on Drive:
 - config.json
 - elderguard_model.onnx
 - model.safetensors
 - tokenizer.json
 - tokenizer_config.json
 - vocab.txt


In [ ]:
!rm - f / content / drive / MyDrive / ElderGuard / model / elderguard_distilbert_v1 / elderguard_model.onnx.data

rm: cannot remove '-': No such file or directory
rm: cannot remove 'f': No such file or directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'content': No such file or directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'drive': Is a directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'MyDrive': No such file or directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'ElderGuard': No such file or directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'model': No such file or directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'elderguard_distilbert_v1': No such file or directory
rm: cannot remove '/': Is a directory
rm: cannot remove 'elderguard_model.onnx.data': No such file or directory


In [ ]:
import os

data_file = "/content/drive/MyDrive/ElderGuard/model/elderguard_distilbert_v1/elderguard_model.onnx.data"
if os.path.exists(data_file):
  os.remove(data_file)
  print("Removed leftover data file.")
else:
  print("Already clean.")

Already clean.
